## AlexNet Training Phase

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from torchvision.models import alexnet
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU available: True
Device: Tesla T4


In [2]:
import os, zipfile, urllib.request

DATA_DIR = "/kaggle/working/tiny-imagenet-200"

if not os.path.exists(DATA_DIR):
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    urllib.request.urlretrieve(url, "/kaggle/working/tiny-imagenet-200.zip")

    with zipfile.ZipFile("/kaggle/working/tiny-imagenet-200.zip", 'r') as zip_ref:
        zip_ref.extractall("/kaggle/working/")

In [3]:
import shutil

val_dir = os.path.join(DATA_DIR, "val")
images_dir = os.path.join(val_dir, "images")
annotations = os.path.join(val_dir, "val_annotations.txt")

with open(annotations) as f:
    for line in f:
        img, cls = line.split()[:2]
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
        shutil.move(
            os.path.join(images_dir, img),
            os.path.join(val_dir, cls, img)
        )

shutil.rmtree(images_dir)

In [ ]:
# define data transformations on training and validation set
def get_loaders():
    normalize = transforms.Normalize(
        (0.4802, 0.4481, 0.3975),
        (0.2302, 0.2265, 0.2262)
    )

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
        transforms.ToTensor(),
        normalize,
        transforms.RandomErasing(p=0.1)
    ])

    val_tf = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        normalize
    ])

    train_ds = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "train"),
        transform=train_tf
    )

    val_ds = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "val"),
        transform=val_tf
    )

    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader

In [ ]:
def get_model():
    # using pre-trained weights from ImageNet
    model = alexnet(weights="IMAGENET1K_V1")

    # replacing classifier head
    model.classifier[6] = nn.Linear(4096, 200)

    # freezing feature extractor
    for param in model.features.parameters():
        param.requires_grad = False

    return model

In [ ]:
# create evaluation fcn
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

In [ ]:
import psutil
import time

# functions for tracking model size, GPU/CPU usage
def get_model_size(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024**2  # MB

def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_gpu_usage():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2  # MB
    return 0

In [ ]:
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader, val_loader = get_loaders()

    model = get_model().to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    # define optimizer
    optimizer = optim.SGD(
        model.classifier.parameters(),
        lr=0.05,
        momentum=0.9,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    scaler = GradScaler()

    best_acc = 0
    total_start_time = time.time()

    print("\n===== TRAINING START =====")

    for epoch in range(50):
        epoch_start = time.time()

        model.train()
        total_loss = 0

        cpu_usage_epoch = []
        gpu_usage_epoch = []

        for x, y in tqdm(train_loader):
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            with autocast():
                out = model(x)
                loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            # track usage
            cpu_usage_epoch.append(get_cpu_usage())
            if torch.cuda.is_available():
                gpu_usage_epoch.append(get_gpu_usage())

        scheduler.step()

        epoch_time = time.time() - epoch_start
        val_acc = evaluate(model, val_loader, device)

        # basic logging
        print(f"\nEpoch {epoch+1:02d}")
        print(f"Loss: {total_loss:.2f}")
        print(f"Val Acc: {val_acc:.2f}%")
        print(f"Epoch Time: {epoch_time:.2f}s")

        # print avg cpu/gpu usage every 5 epochs
        if (epoch + 1) % 5 == 0:
            print(f"Avg CPU Usage: {sum(cpu_usage_epoch)/len(cpu_usage_epoch):.2f}%")
            if gpu_usage_epoch:
                print(f"Avg GPU Memory: {sum(gpu_usage_epoch)/len(gpu_usage_epoch):.2f} MB")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "/kaggle/working/best_model.pth")

    total_time = time.time() - total_start_time

    print("\n===== TRAINING COMPLETE =====")
    print(f"Total Training Time: {total_time/60:.2f} minutes")
    print(f"Best Validation Accuracy: {best_acc:.2f}%")
    print(f"Model Size: {get_model_size(model):.2f} MB")

    return model, val_loader

In [ ]:
# train model
model, val_loader = train()

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 226MB/s] 
/tmp/ipykernel_55/1903672054.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



===== TRAINING START =====


  0%|          | 0/782 [00:00<?, ?it/s]/tmp/ipykernel_55/1903672054.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 782/782 [06:42<00:00,  1.94it/s]



Epoch 01
Loss: 3009.87
Val Acc: 26.19%
Epoch Time: 402.44s


...


Epoch 48
Loss: 2077.29
Val Acc: 50.83%
Epoch Time: 387.80s


100%|██████████| 782/782 [06:29<00:00,  2.01it/s]



Epoch 49
Loss: 2075.99
Val Acc: 50.91%
Epoch Time: 389.76s


100%|██████████| 782/782 [06:45<00:00,  1.93it/s]



Epoch 50
Loss: 2068.65
Val Acc: 50.87%
Epoch Time: 405.81s
Avg CPU Usage: 65.56%
Avg GPU Memory: 734.81 MB

===== TRAINING COMPLETE =====
Total Training Time: 343.53 minutes
Best Validation Accuracy: 50.96%
Model Size: 220.58 MB


## AlexNet INT8 Quantization Phase

In [10]:
print("\n===== INT8 QUANTIZATION =====")

model_cpu = model.to("cpu").float()
model_cpu.eval()

start_time = time.time()

quantized_model = torch.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

acc = evaluate(quantized_model, val_loader, "cpu")

print(f"INT8 Accuracy: {acc:.2f}%")
print(f"INT8 Model Size: {get_model_size(quantized_model):.2f} MB")
print(f"INT8 CPU Usage: {get_cpu_usage():.2f}%")
print(f"INT8 GPU Usage: {get_gpu_usage():.2f} MB")
print(f"Quantization Time: {time.time() - start_time:.2f}s")


===== INT8 QUANTIZATION =====


/tmp/ipykernel_55/4128840117.py:8: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


INT8 Accuracy: 50.81%
INT8 Model Size: 9.42 MB
INT8 CPU Usage: 36.00%
INT8 GPU Usage: 17.25 MB
Quantization Time: 109.87s
